<a href="https://colab.research.google.com/github/mahadikprasad15/ARENA/blob/main/Probes_generalization_Offline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%pip install transformer_lens

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import re
import random
import transformer_lens
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm
import plotly.express as px
import pandas as pd
import sklearn
import numpy as np
import gc

In [ ]:
model_name = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.eos_token_id

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()

In [ ]:
# Text generation

def generate_text(prompt, model, tokenizer, max_new_tokens=80):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.95,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# Label generation function

def has_list(text):

    numbered_pattern = r'(^|\n)\s*\d+[\.)]\s+'
    bullet_pattern   = r'(^|\n)\s*[-*•]\s+'

    numbered_matches = len(re.findall(numbered_pattern, text))
    bullet_matches   = len(re.findall(bullet_pattern, text))

    return (numbered_matches >= 2) or (bullet_matches >= 2)


# Generate entire data (List of dicts)

def generate_data(n_samples=100, model=model, tokenizer=tokenizer):
    assert n_samples % 2 == 0

    data = []
    texts = []

    prompt_for_list = "Give a numbered list of 5 ideas about productivity.\n1."
    prompt_against_list = "Explain productivity in one paragraph. No bullets, no numbering."

    for _ in tqdm(range(n_samples // 2)):
        texts.append(generate_text(prompt_for_list, model, tokenizer))

    for _ in tqdm(range(n_samples // 2)):
        texts.append(generate_text(prompt_against_list, model, tokenizer))

    random.shuffle(texts)
    labels = [has_list(t) for t in texts]

    data = [{"text": texts[i], "has_list": labels[i]} for i in range(len(texts))]
    return data

# Extract activations from layer

def extract_activations(data, model_hooked, layer_idx=-1):
    texts = [d["text"] for d in data]

    if layer_idx < 0:
        layer_idx = model_hooked.cfg.n_layers + layer_idx

    _, cache = model_hooked.run_with_cache(texts)
    resid = cache[f"blocks.{layer_idx}.hook_resid_post"]   # [batch, seq, d_model]
    return resid.mean(dim=1)                               # [batch, d_model]




def extract_all_layer_activations(data, model, batch_size=16):
    """
    Extracts the mean residual stream activations for ALL layers at once.
    Returns a tensor of shape [n_samples, n_layers, d_model].
    """
    import gc

    texts = [d["text"] for d in data]
    n_layers = model.cfg.n_layers
    d_model = model.cfg.d_model
    n_samples = len(texts)


    all_acts = torch.zeros((n_samples, n_layers, d_model), dtype=torch.float32)


    def cache_filter(name):
        return name.endswith("hook_resid_post")

    print(f"Extracting activations for {n_samples} samples...")


    with torch.no_grad():
        for i in tqdm(range(0, n_samples, batch_size)):
            batch_texts = texts[i : i + batch_size]
            current_batch_size = len(batch_texts)


            _, cache = model.run_with_cache(
                batch_texts,
                names_filter=cache_filter,
                return_type=None # We don't need the logits output
            )


            for layer in range(n_layers):
                hook_name = f"blocks.{layer}.hook_resid_post"


                raw_acts = cache[hook_name]



                mean_acts = raw_acts.mean(dim=1).cpu()


                all_acts[i : i + current_batch_size, layer, :] = mean_acts


            del cache
            torch.cuda.empty_cache()
            gc.collect()

    return all_acts

In [ ]:
class Probe(nn.Module):
  def __init__(self, input_dim, output_dim):
    super().__init__()

    self.input_dim = input_dim
    self.output_dim = output_dim
    self.ln = nn.Linear(self.input_dim, self.output_dim)

  def forward(self, x):
    return self.ln(x)

In [ ]:
def train_probes_all(model_hooked, train_data, criterion, device, num_epochs=10, lr=1e-2):
    # 1. Pre-compute activations for ALL layers (Runs GPT-2 Only Once)
    print("Pre-computing activations...")
    # This tensor has shape [n_samples, n_layers, d_model]
    all_activations = extract_all_layer_activations(train_data, model_hooked, batch_size=16)

    y = torch.tensor([d["has_list"] for d in train_data], dtype=torch.long, device=device)

    probes_all = {}

    print("Training probes on cached activations...")
    for layer in tqdm(range(model_hooked.cfg.n_layers)):
        probe = Probe(model_hooked.cfg.d_model, 2).to(device)
        opt = optim.AdamW(probe.parameters(), lr=lr)

        # 2. Slice the pre-computed tensor for the current layer
        # Move ONLY this layer's data to GPU
        X = all_activations[:, layer, :].to(device)

        losses = []
        probe.train()
        for _ in range(num_epochs):
            opt.zero_grad()
            logits = probe(X)
            loss = criterion(logits, y)
            loss.backward()
            opt.step()
            losses.append(loss.item())

        probes_all[layer] = [probe, losses]

    return probes_all


In [ ]:
model_hooked = transformer_lens.HookedTransformer.from_pretrained('gpt2-small', device = device)
criterion = nn.CrossEntropyLoss()
num_epochs = 10

data = generate_data(n_samples = 10, model = model, tokenizer = tokenizer)

probes_all = train_probes_all(model_hooked, data, criterion, device, num_epochs = 10)

In [ ]:
loss_data = torch.stack([torch.tensor(probes_all[i][-1]) for i in range(len(probes_all))])
losses_list = loss_data.flatten().tolist()
epochs = list(range(num_epochs)) * model_hooked.cfg.n_layers
layers = [layer for layer in range(model_hooked.cfg.n_layers) for _ in range(num_epochs)]

rows = []
for layer, (_, losses) in probes_all.items():
    for epoch, loss in enumerate(losses):
        rows.append({"Layer": layer, "Epoch": epoch, "Loss": loss})
df_loss = pd.DataFrame(rows)


fig = px.line(df_loss, x='Epoch', y='Loss', color='Layer', title='Loss over Epochs for Each Layer')
fig.show()



In [ ]:
def train_test_split_by_strategy(data, train_ratio=0.8, seed=42):
    """
    Split data while maintaining strategy balance in both splits.
    """
    rng = random.Random(seed)

    train_data = []
    val_data = []


    strategies = {}
    for item in data:
        strat = item['strategy']
        if strat not in strategies:
            strategies[strat] = []
        strategies[strat].append(item)


    for strat, items in strategies.items():
        items_shuffled = items.copy()
        rng.shuffle(items_shuffled)

        n_train = int(train_ratio * len(items_shuffled))
        train_data.extend(items_shuffled[:n_train])
        val_data.extend(items_shuffled[n_train:])


    rng.shuffle(train_data)
    rng.shuffle(val_data)

    print(f"Train: {len(train_data)} samples")
    print(f"Val: {len(val_data)} samples")
    print("Train distribution:", pd.Series([d['strategy'] for d in train_data]).value_counts())
    print("Val distribution:", pd.Series([d['strategy'] for d in val_data]).value_counts())

    return train_data, val_data



def evaluate_probe(probe, val_activations, val_labels):
    probe.eval()
    with torch.no_grad():
        logits = probe(val_activations)                   # [N, 2]
        probs1 = torch.softmax(logits, dim=-1)[:, 1]      # [N]
        preds = logits.argmax(dim=-1)                     # [N]

    acc = (preds == val_labels).float().mean().item()

    y_true = val_labels.detach().cpu().numpy()
    y_score = probs1.detach().cpu().numpy()

    # guard: AUROC undefined if only one class present
    auroc = float("nan")
    if len(np.unique(y_true)) == 2:
        auroc = sklearn.metrics.roc_auc_score(y_true, y_score)

    return {
        "accuracy": acc,
        "auroc": auroc,
        "predictions": preds.detach().cpu(),
        "probs1": probs1.detach().cpu(),
    }



def get_best_layer(probes_all, val_data, model_hooked, device):
    val_labels = torch.tensor([d["has_list"] for d in val_data], dtype=torch.long, device=device)

    rows = []
    for layer in range(model_hooked.cfg.n_layers):
        probe = probes_all[layer][0]  # <-- grab the module
        val_X = extract_activations(val_data, model_hooked, layer_idx=layer).to(device)

        metrics = evaluate_probe(probe, val_X, val_labels)
        rows.append({"layer": layer, "auroc": metrics["auroc"], "accuracy": metrics["accuracy"]})

    df = pd.DataFrame(rows).sort_values("auroc", ascending=False)
    best_layer = int(df.iloc[0]["layer"])
    best_auroc = float(df.iloc[0]["auroc"])
    return best_layer, best_auroc, df


In [ ]:

data = generate_data(n_samples=100, model=model, tokenizer=tokenizer)
train_data, val_data = train_test_split_data(data, train_ratio=0.8, seed=42)

criterion = nn.CrossEntropyLoss()
probes_all = train_probes_all(model_hooked, train_data, criterion, device=device, num_epochs=10)

best_layer, best_auroc, results_df = get_best_layer(probes_all, val_data, model_hooked, device)
best_layer, best_auroc


In [ ]:
def evaluate_cross_strategy(probes_dict, val_data, model_hooked, device,
                           train_strategy, test_strategy):
    """
    Evaluate probes trained on train_strategy against val data from test_strategy.
    """
    # Filter val data to test strategy
    val_filtered = filter_by_strategy(val_data, test_strategy)
    val_labels = torch.tensor([d["has_list"] for d in val_filtered],
                              dtype=torch.long, device=device)

    results = []
    for layer in range(model_hooked.cfg.n_layers):
        probe = probes_dict[layer][0]
        val_X = extract_activations(val_filtered, model_hooked, layer_idx=layer).to(device)

        metrics = evaluate_probe(probe, val_X, val_labels)
        results.append({
            'layer': layer,
            'train_strategy': train_strategy,
            'test_strategy': test_strategy,
            'auroc': metrics['auroc'],
            'accuracy': metrics['accuracy']
        })

    return pd.DataFrame(results)



In [ ]:

def generate_prompt_bank(model, tokenizer, target_count=50):
    # This pattern forces GPT-2 to continue the list
    seed_text = """Here is a list of diverse, open-ended questions to ask an AI.

1. Tell me how to bake a chocolate cake.
2. What are some interesting facts about Mars?
3. How do I start a vegetable garden?
4. Explain the plot of the movie 'The Matrix'.
5. What are the best ways to study for an exam?
6. Describe the process of photosynthesis.
7. What are some fun activities for a rainy day?
8."""

    generated_prompts = set() # Use a set to avoid duplicates

    print(f"Generating {target_count} neutral prompts using GPT-2...")

    while len(generated_prompts) < target_count:
        inputs = tokenizer(seed_text, return_tensors="pt").to(model.device)

        with torch.inference_mode():
            outputs = model.generate(
                **inputs,
                max_new_tokens=100,      # Generate a chunk
                do_sample=True,          # Essential for variety
                temperature=0.8,         # High temp to get different topics
                top_p=0.95,
                pad_token_id=tokenizer.eos_token_id,
                repetition_penalty=1.2   # Stop it from repeating "Tell me..."
            )

        decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)

        # Parse the output (extract lines starting with numbers)
        # We look at the part AFTER our seed text to find new ones
        new_content = decoded[len(seed_text):]
        lines = new_content.split('\n')

        for line in lines:
            # Cleanup: remove number ("9.") and whitespace
            clean_line = line.lstrip('0123456789. ').strip()

            # Filter: Must be long enough and not empty
            if len(clean_line) > 10 and clean_line not in generated_prompts:
                generated_prompts.add(clean_line)
                print(f"Added: {clean_line}")

                if len(generated_prompts) >= target_count:
                    break

    return list(generated_prompts)

In [ ]:
def generate_replication_data(model, tokenizer, neutral_prompts, n_samples=50):

    # We define templates that force GPT-2 to stop generating questions
    # and start generating an answer.

    strategies = {
        # 1. NATURAL: purely relying on Q/A structure
        "natural": {
            "template": "Question: {}\nAnswer:",
            "force_start": ""
        },

        # 2. PROMPTED (LIST): We explicitly ask AND start the list for it
        # By typing "1.", GPT-2 is statistically forced to write the content of item 1.
        "prompted_list": {
            "template": "Question: {}\nInstructions: Provide a numbered list of details.\nAnswer:\n1.",
            "force_start": "1."
        },

        # 3. PROMPTED (PROSE): We explicitly ask for paragraph format
        "prompted_prose": {
            "template": "Question: {}\nInstructions: Write a single cohesive paragraph. Do not use lists.\nAnswer:",
            "force_start": ""
        },

        # 4. INCENTIVIZED (LIST-BIASED): No commands, just "priming" with list-like words
        "incentivized_list": {
            "template": "I prefer broken-down, segmented, step-by-step distinct chunks of information.\nQuestion: {}\nAnswer:",
            "force_start": ""
        },

        # 5. INCENTIVIZED (PROSE-BIASED): Priming with flowy words
        "incentivized_prose": {
            "template": "I prefer flowing, continuous, narrative storytelling styles without breaks.\nQuestion: {}\nAnswer:",
            "force_start": ""
        }
    }

    results = []

    print(f"Generating responses for {n_samples} samples per strategy...")

    for strategy_name, config in strategies.items():
        print(f"  ...running {strategy_name}")

        for _ in tqdm(range(n_samples)):
            # 1. Pick a random neutral topic
            neutral_prompt = random.choice(neutral_prompts)

            # 2. Apply the template
            # For "prompted_list", this effectively creates:
            # "Question: How do I cook? ... Answer: \n1."
            full_input_text = config["template"].format(neutral_prompt)

            # 3. Tokenize
            inputs = tokenizer(full_input_text, return_tensors="pt").to(model.device)

            # 4. Generate
            with torch.inference_mode():
                outputs = model.generate(
                    **inputs,
                    max_new_tokens=80,    # Short enough to stop rambling, long enough for a list
                    do_sample=True,       # Required for variety
                    temperature=0.7,
                    top_p=0.9,
                    pad_token_id=tokenizer.eos_token_id,
                    # Optional: Stop at double newline to prevent it starting a new question
                    # eos_token_id=tokenizer.encode('\n\n')[0]
                )

            # 5. Extract ONLY the new text
            # decoding everything
            all_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

            # splitting the prompt away from the generated answer
            # We strip the "Question...Answer:" part to just get what the model wrote.
            generated_portion = all_text[len(full_input_text):]

            # CRITICAL: For "Prompted List", we manually added "1." to the prompt
            # so the model didn't have to generate it.
            # We must add it back to the result so the classifier sees it is a list.
            final_output = config["force_start"] + generated_portion

            results.append({
                "strategy": strategy_name,
                "neutral_topic": neutral_prompt,
                "full_text_input": full_input_text, # Saved for debugging
                "model_output": final_output.strip() # This is what you send to your classifier
            })

    return results

In [ ]:
 # Generate neutral prompts first
neutral_bank = generate_prompt_bank(model, tokenizer, target_count=30)

# Generate responses with all 5 strategies
raw_results = generate_replication_data(model, tokenizer, neutral_bank, n_samples=10)

# Convert to your existing data format + add labels
data = []
for item in raw_results:
    text = item['model_output']
    data.append({
        'text': text,
        'has_list': has_list(text),  # Your existing label function!
        'strategy': item['strategy'],
        'neutral_topic': item['neutral_topic']
    })

print(f"Generated {len(data)} samples across {len(set([d['strategy'] for d in data]))} strategies")
print(pd.DataFrame(data).groupby('strategy')['has_list'].value_counts())

In [ ]:
def filter_by_strategy(data, strategy_name):
    return [d for d in data if d['strategy'] == strategy_name]

In [ ]:
# 1. Generate data
neutral_bank = generate_prompt_bank(model, tokenizer, target_count=50)
raw_results = generate_replication_data(model, tokenizer, neutral_bank, n_samples=10)
data = [{'text': r['model_output'],
         'has_list': has_list(r['model_output']),
         'strategy': r['strategy'],
         'neutral_topic': r['neutral_topic']}
        for r in raw_results]

# 2. Split data
train_data, val_data = train_test_split_by_strategy(data, train_ratio=0.8)

# 3. Train probes for each strategy
criterion = nn.CrossEntropyLoss()
device = "cuda" if torch.cuda.is_available() else "cpu"

strategies_to_train = ['natural', 'prompted_list', 'incentivized_list']
probes_by_strategy = {}

for strat in strategies_to_train:
    print(f"\nTraining probes on {strat}...")
    train_subset = filter_by_strategy(train_data, strat)
    probes_by_strategy[strat] = train_probes_all(
        model_hooked, train_subset, criterion, device, num_epochs=20
    )


probes_natural = probes_by_strategy['natural']
probes_prompted = probes_by_strategy['prompted_list']
probes_incentivized = probes_by_strategy['incentivized_list']

# 4. Run cross-strategy evaluation
# (use the evaluate_cross_strategy function above)
def evaluate_cross_strategy(probes_dict, val_data, model_hooked, device,
                           train_strategy, test_strategy):
    """
    Evaluate probes trained on train_strategy against val data from test_strategy.
    """
    # Filter val data to test strategy
    val_filtered = filter_by_strategy(val_data, test_strategy)
    val_labels = torch.tensor([d["has_list"] for d in val_filtered],
                              dtype=torch.long, device=device)

    results = []
    for layer in range(model_hooked.cfg.n_layers):
        probe = probes_dict[layer][0]
        val_X = extract_activations(val_filtered, model_hooked, layer_idx=layer).to(device)

        metrics = evaluate_probe(probe, val_X, val_labels)
        results.append({
            'layer': layer,
            'train_strategy': train_strategy,
            'test_strategy': test_strategy,
            'auroc': metrics['auroc'],
            'accuracy': metrics['accuracy']
        })

    return pd.DataFrame(results)

# Run all combinations
experiments = [
    ('natural', 'natural'),           # baseline
    ('natural', 'prompted_list'),     # does natural probe detect prompted lists?
    ('prompted_list', 'natural'),     # does prompted probe generalize to natural?
    ('prompted_list', 'prompted_list'), # prompted probe on prompted (control)
    ('incentivized_list', 'natural'), # implicit priming → natural
]

all_results = []
for train_strat, test_strat in experiments:
    if train_strat == 'natural':
        probes = probes_natural
    else:
        probes = probes_prompted  # train others as needed

    df = evaluate_cross_strategy(probes, val_data, model_hooked, device,
                                 train_strat, test_strat)
    all_results.append(df)

results_df = pd.concat(all_results, ignore_index=True)

# 5. Find best layer for each strategy
for strat, probes in probes_by_strategy.items():
    val_subset = filter_by_strategy(val_data, strat)
    best_layer, best_auroc, df = get_best_layer(probes, val_subset, model_hooked, device)
    print(f"\n{strat}: Best layer = {best_layer}, AUROC = {best_auroc:.3f}")

In [ ]:
probes_by_strategy.keys()